In [2]:
# Exemplo: ∫_0^1 x^2 dx usando a regra 1/3 de Simpson, com m=2
import numpy as np
from fractions import Fraction

# Dados do exemplo
f = lambda x: x**2
a, b = 0.0, 1.0
m = 2                   # número de subintervalos (PAR)
n = m + 1               # número de pontos
x = np.linspace(a, b, n)
h = (b - a) / m        # deve dar 0.5

# Pontos e valores
x0, x1, x2 = x
f0, f1, f2 = f(x0), f(x1), f(x2)

# Simpson 1/3: h/3 [ f(x0) + 4 f(x1) + f(x2) ]
I_simpson = h/3 * (f0 + 4*f1 + f2)
I_exact = 1.0/3.0

print("Exemplo: Calcule ∫_0^1 x^2 dx usando a regra 1/3 de Simpson, com m=2.\n")

print("Solução\nExata:")
print("∫_0^1 x^2 dx = x^3/3 |^1_0 = 1/3\n")

print("Aproximada (Simpson 1/3):")
print(f"h = {h}")
print("I ≈ h/3 [ f(x0) + 4 f(x1) + f(x2) ]")
print(f"  = {h}/3 [ f(0) + 4 f(0.5) + f(1) ]")
print(f"  = {h}/3 [ {f0} + 4*({x1}^2) + {f2} ]")
inside = f0 + 4*f1 + f2
print(f"  = {h}/3 [ {inside} ]")
print(f"  = {I_simpson:.10f} ≈ {Fraction(I_simpson).limit_denominator()}\n")

print(f"Integral exata = {I_exact} = {Fraction(I_exact).limit_denominator()}")
print(f"Erro absoluto = {abs(I_exact - I_simpson):.3e}")
print(f"Erro relativo = {abs(I_exact - I_simpson)/I_exact:.3e}")

Exemplo: Calcule ∫_0^1 x^2 dx usando a regra 1/3 de Simpson, com m=2.

Solução
Exata:
∫_0^1 x^2 dx = x^3/3 |^1_0 = 1/3

Aproximada (Simpson 1/3):
h = 0.5
I ≈ h/3 [ f(x0) + 4 f(x1) + f(x2) ]
  = 0.5/3 [ f(0) + 4 f(0.5) + f(1) ]
  = 0.5/3 [ 0.0 + 4*(0.5^2) + 1.0 ]
  = 0.5/3 [ 2.0 ]
  = 0.3333333333 ≈ 1/3

Integral exata = 0.3333333333333333 = 1/3
Erro absoluto = 0.000e+00
Erro relativo = 0.000e+00


In [3]:
import math
from typing import Callable, Tuple, List

def simpson13_composta_latex(
    f: Callable[[float], float],
    a: float,
    b: float,
    m: int,                 # número de subintervalos (precisa ser PAR)
    func_name: str = r"f(x)"
) -> Tuple[float, float, List[float], List[float], str]:
    """
    Regra 1/3 de Simpson composta + string LaTeX para usar com display(Math(...)).

    f         : função Python (ex: math.exp)
    a, b      : limites de integração
    m         : número de subintervalos (precisa ser PAR)
    func_name : texto que aparece no integrando (ex: 'e^x', 'f(x)')

    Retorna: (aprox, h, xs, fs, latex_str)
    """
    if m % 2 != 0:
        raise ValueError("Na regra 1/3 de Simpson, o número de subintervalos m deve ser par.")

    h = (b - a) / m
    xs = [a + i*h for i in range(m + 1)]
    fs = [f(x) for x in xs]

    # Soma dos termos com peso 4 (índices ímpares)
    soma_4 = sum(fs[i] for i in range(1, m, 2))
    # Soma dos termos com peso 2 (índices pares internos)
    soma_2 = sum(fs[i] for i in range(2, m, 2))

    aprox = (h/3) * (fs[0] + fs[-1] + 4*soma_4 + 2*soma_2)

    # ---------- Construção do LaTeX ----------

    # Parte simbólica: f(x0) + f(xm) + 4[f(x1)+f(x3)+...] + 2[f(x2)+f(x4)+...]
    sym_endpoints = r"{f}\big({x0}\big) + {f}\big({xm}\big)".format(
        f=func_name, x0=f"{xs[0]:.3g}", xm=f"{xs[-1]:.3g}"
    )

    sym_4_terms = " + ".join(
        [fr"{func_name}\big({xs[i]:.3g}\big)" for i in range(1, m, 2)]
    )
    sym_2_terms = " + ".join(
        [fr"{func_name}\big({xs[i]:.3g}\big)" for i in range(2, m, 2)]
    )

    if sym_4_terms:
        sym_4_block = r"4\big[" + sym_4_terms + r"\big]"
    else:
        sym_4_block = ""

    if sym_2_terms:
        sym_2_block = r"2\big[" + sym_2_terms + r"\big]"
    else:
        sym_2_block = ""

    symbolic_parts = [sym_endpoints]
    if sym_4_block:
        symbolic_parts.append(sym_4_block)
    if sym_2_block:
        symbolic_parts.append(sym_2_block)
    symbolic = " + ".join(symbolic_parts)

    # Parte numérica: f(x0)=..., f(x1)=..., etc
    num_endpoints = f"{fs[0]:.6g} + {fs[-1]:.6g}"
    num_4_terms = " + ".join([fr"{fs[i]:.6g}" for i in range(1, m, 2)])
    num_2_terms = " + ".join([fr"{fs[i]:.6g}" for i in range(2, m, 2)])

    num_parts = [num_endpoints]
    if num_4_terms:
        num_parts.append(r"4\big[" + num_4_terms + r"\big]")
    if num_2_terms:
        num_parts.append(r"2\big[" + num_2_terms + r"\big]")
    numeric = " + ".join(num_parts)

    a_s = f"{a:.3g}"
    b_s = f"{b:.3g}"
    h_s = f"{h:.4g}"
    aprox_s = f"{aprox:.9f}"

    latex_str = (
        r"\int_{" + a_s + "}^{" + b_s + "} " + func_name + r"\,dx"
        + r"\approx \frac{" + h_s + r"}{3}\left[" + symbolic + r"\right]"
        + r" = \frac{" + h_s + r"}{3}\left[" + numeric + r"\right]"
        + r" = " + aprox_s
    )

    return aprox, h, xs, fs, latex_str


In [22]:
from IPython.display import Math, display
import math

def f_exp(x):
    return math.exp(-(x/2)**2)

a, b = 0.0, 2.0
m = 10

aprox, h, xs, fs, latex_str = simpson13_composta_latex(
    f_exp, a, b, m,
    func_name=r"e^{-(x/2)^2}"
)

print(f"Intervalo [{a}, {b}] com m={m} subintervalos (h={h})")
print(f"Pontos: {', '.join(f'{x:.1f}' for x in xs)}")
print(f"Aproximação = {aprox:.9f}")

# Renderiza expressão LaTeX mostrando a fórmula aplicada
display(Math(latex_str))


Intervalo [0.0, 2.0] com m=10 subintervalos (h=0.2)
Pontos: 0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0
Aproximação = 1.493649897


<IPython.core.display.Math object>